# 02 · Load everything in parallel, validate every record at the boundary

**Produces:** `data/clean/bookings_all.csv.gz` (one row per booking-test, both systems, normalised), `data/clean/bookings.csv.gz` (one row per booking), `data/clean/rejects_bookings.csv` (every rejected record with its reason).

**Needs:** notebook 01 run; the pack in `data/raw/`.

Two ideas in this notebook. First, the pack is a few hundred files, so the unit of work is a file and a process pool reads them side by side. Second, nothing enters the analysis without passing a model that says what a booking must look like; what fails is kept, counted and listed, so input equals accepted plus rejected.

## CONFIG

In [1]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

3.0.6 2.5.4


## 1 · MAP: two systems, one grain

Legacy exports are one row per booking-test. Nova exports are one JSON object per booking with the tests nested. Both are brought to booking-test grain by the loaders, then rolled up to booking grain here. The picture:

```
legacy/*.csv.gz  ─┐                           ┌─ bookings_all (booking-test grain)
                  ├─ process pool ─ concat ───┤
nova/*.jsonl.gz  ─┘        │                  └─ bookings (booking grain, n_tests, amount)
                           └─ pydantic validation ─ rejects_bookings
```

## 2 · DO: sequential against parallel, timed, so the choice is a measured one

In [3]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
print(len(legacy_files), "legacy files,", len(nova_files), "nova files")

t0 = time.time()
seq = pd.concat([loaders.read_legacy_file(p) for p in legacy_files[:12]], ignore_index=True)
t_seq = time.time() - t0
t0 = time.time()
par = loaders.read_many(legacy_files[:12], "legacy", workers=CONFIG["workers"])
t_par = time.time() - t0
print(f"12 files: sequential {t_seq:.1f}s, process pool {t_par:.1f}s, same rows: {len(seq) == len(par)}")

102 legacy files, 6 nova files
12 files: sequential 0.8s, process pool 1.1s, same rows: True


> **Interview angle.** Why a process pool and not a thread pool here? When would threads be the right choice?
>
> *What a good answer covers:* Decompressing gzip, parsing CSV and parsing JSON are CPU-bound Python work, and the global interpreter lock lets only one thread run Python bytecode at a time, so threads would queue. Processes each have their own interpreter. Threads win when the work waits on I/O, such as network calls or disk reads of already-parsed data, and when the data to pass between workers is large, since processes copy it.

## 3 · DO: the full load

In [4]:
t0 = time.time()
legacy = loaders.read_many(legacy_files, "legacy", workers=CONFIG["workers"])
print(f"legacy: {len(legacy):,} booking-test rows from {len(legacy_files)} files in {time.time()-t0:.1f}s; repeated headers dropped: {legacy.attrs['repeated_header_lines']}")
t0 = time.time()
nova = loaders.read_many(nova_files, "nova", workers=CONFIG["workers"]) if nova_files else pd.DataFrame()
print(f"nova: {len(nova):,} booking-test rows from {len(nova_files)} files in {time.time()-t0:.1f}s")

legacy: 1,252,765 booking-test rows from 102 files in 2.9s; repeated headers dropped: 21
nova: 62,418 booking-test rows from 6 files in 0.8s


In [5]:
# Concatenating two frames whose columns differ: name the difference before you union, so a silent NaN column cannot surprise you later
only_legacy = set(legacy.columns) - set(nova.columns)
only_nova = set(nova.columns) - set(legacy.columns)
print("only in legacy:", only_legacy, "| only in nova:", only_nova)
for c in only_nova: legacy[c] = pd.NA
for c in only_legacy: nova[c] = pd.NA
all_rows = pd.concat([legacy, nova[legacy.columns]], ignore_index=True)
all_rows["city"] = all_rows["city"].astype("string").replace({"Bangalore": "Bengaluru"})
print(f"{len(all_rows):,} booking-test rows; {all_rows['booking_id'].nunique():,} distinct bookings")
all_rows.groupby("system", observed=True).agg(rows=("booking_id", "size"), bookings=("booking_id", "nunique"), first=("slot_at", "min"), last=("slot_at", "max"))

only in legacy: set() | only in nova: {'status_at_export', 'rescheduled_from'}
1,315,183 booking-test rows; 567,986 distinct bookings


,rows,bookings,first,last
system,,,,
legacy,1252765,541464,2025-01-01 06:00:00,2026-06-30 19:59:00
nova,62418,26522,2026-04-01 06:01:00,2026-07-04 10:50:00


### SEE: the grain trap, in numbers

The table above is the whole of brief B in one line: rows per booking differ by system, so a row count across the migration is not a booking count. Nova also carries rescheduled records. Decide now what a booking is, write the rule, and apply it in notebook 03.

## 4 · DO: validate every booking with a model

`models.Booking` says what one visit must look like. Build one record per booking from the rows, validate each, and keep the rejects with reasons. This is slower than a vectorised check and that is the point of doing it once at the boundary: the model is the contract, and the rejects file is the evidence that nothing slipped past it.

In [6]:
def records_from_rows(rows: pd.DataFrame):
    """Group booking-test rows into one dict per booking, the shape the model expects.

    Sorted once, then sliced by the boundaries np.unique returns: a few seconds for a million rows,
    where a groupby.apply with a Python lambda would take minutes. Same result, measured difference."""
    cols = ["booking_id", "system", "created_at", "slot_at", "city", "clinic_id", "channel", "source", "patient_id", "home_fee", "rescheduled_from", "status_at_export"]
    r = rows.sort_values("booking_id", kind="stable").reset_index(drop=True)
    ids = r["booking_id"].to_numpy()
    _, starts = np.unique(ids, return_index=True)
    ends = np.append(starts[1:], len(r))
    meta = r[cols].to_dict("records")
    codes = r["test_code"].astype("string").fillna("").to_numpy()
    prices = r["test_price"].fillna(0).astype(int).to_numpy()
    for a, b in zip(starts, ends):
        rec = {k: (None if (v is pd.NA or (isinstance(v, float) and v != v)) else v) for k, v in meta[a].items()}
        rec["tests"] = [{"code": c, "price": int(p)} for c, p in zip(codes[a:b], prices[a:b])]
        yield rec

t0 = time.time()
accepted, rejects = models.validate_many(models.Booking, records_from_rows(all_rows))
print(f"validated {len(accepted)+len(rejects):,} bookings in {time.time()-t0:.0f}s (Python-level validation, once, at the boundary): accepted {len(accepted):,}, rejected {len(rejects):,}")
checks.reconciles("booking validation", all_rows["booking_id"].nunique(), len(accepted), len(rejects))

validated 567,986 bookings in 30s (Python-level validation, once, at the boundary): accepted 564,879, rejected 3,107
PASS  booking validation: input = kept + rejected   567,986 = 564,879 + 3,107


True

In [8]:
rej = pd.DataFrame([{"booking_id": r["record"].get("booking_id"), "system": r["record"].get("system"),
                     "reason": "; ".join(f"{f}: {m}" for f, m in r["errors"])} for r in rejects])
rej.to_csv(CONFIG["clean"] / "rejects_bookings.csv", index=False)
rej["reason"].str.split(":").str[0].value_counts() if len(rej) else print("no rejects")

reason
model           1027
booking_id       793
tests.0.code     583
tests.1.code     378
tests.2.code     222
tests.3.code      80
tests.4.code      18
tests.5.code       6
Name: count, dtype: int64

> **Interview angle.** What does a pydantic model give you that a pandas `.str.match` does not, and what does it cost?
>
> *What a good answer covers:* A single, named contract for a record that normalises what is safe to normalise and refuses the rest with a per-field reason, reusable by the SQL loader, the API and the tests. It costs time per record (Python-level validation) and a second representation of the data, so you run it once at the boundary and vectorise everything after.

## 5 · DO: the booking-grain table

In [11]:
bookings = pd.DataFrame([{**b.model_dump(exclude={"tests"}), "n_tests": b.n_tests, "amount_rs": b.amount_rs,
                          "has_package": any(t.code.startswith("PK") for t in b.tests)} for b in accepted])
for c in ("created_at", "slot_at"):
    bookings[c] = pd.to_datetime(bookings[c]).dt.tz_localize(None)
bookings["month"] = bookings["slot_at"].dt.to_period("M").astype(str)
bookings["is_reschedule"] = bookings["rescheduled_from"].notna()
print(bookings.shape)
print(bookings.head())
bookings.groupby(["system", "is_reschedule"]).size()

(564879, 17)
             booking_id  system          created_at             slot_at       city clinic_id channel  source   patient_id  home_fee rescheduled_from  \
0  BK-AHM-202501-000001  legacy 2025-01-08 03:16:00 2025-01-09 08:26:00  Ahmedabad    AHM-03  clinic  walkin  PAHM0028525         0              NaN   
1  BK-AHM-202501-000002  legacy 2025-01-03 05:18:00 2025-01-06 13:22:00  Ahmedabad    AHM-04  clinic     app  PAHM0031384         0              NaN   
2  BK-AHM-202501-000003  legacy 2025-01-27 10:46:00 2025-01-28 13:37:00  Ahmedabad    AHM-01    home     app  PAHM0027001       150              NaN   
3  BK-AHM-202501-000004  legacy 2025-01-20 02:48:00 2025-01-24 07:15:00  Ahmedabad    AHM-03  clinic     app  PAHM0007069         0              NaN   
4  BK-AHM-202501-000005  legacy 2025-01-06 08:21:00 2025-01-11 09:56:00  Ahmedabad    AHM-03  clinic     app  PAHM0002520         0              NaN   

  status_at_export  n_tests  amount_rs  has_package    month  is_reschedul

system  is_reschedule
legacy  False            538357
nova    False             25022
        True               1500
dtype: int64

In [8]:
all_rows.to_csv(CONFIG["clean"] / "bookings_all.csv.gz", index=False, compression="gzip")
bookings.to_csv(CONFIG["clean"] / "bookings.csv.gz", index=False, compression="gzip")
checks.check("booking table has one row per booking id", bookings["booking_id"].is_unique, len(bookings))
checks.check("n_tests per booking is at least 1", bool((bookings["n_tests"] >= 1).all()))
checks.summary()

PASS  booking table has one row per booking id   564879
PASS  n_tests per booking is at least 1
3 of 3 checks passed


,check,passed,detail
0,booking validation: input = kept + rejected,True,"567,986 = 564,879 + 3,107"
1,booking table has one row per booking id,True,564879
2,n_tests per booking is at least 1,True,


## 6 · SUM

You now hold every booking from both systems in one table at one grain, with every rejected record listed. Record in `docs/decisions-log.md`: the repeated-header count, the city spelling rule, the reject count and reasons. Record in `docs/approach-log.md`: why booking grain and not booking-test grain for the headline, and what the alternative would have changed.

## Before you move on

- What is a booking? One sentence. Does a rescheduled nova record count as a second booking?
- How many bookings did the validator reject, and could any of them have been real visits?
- If a reviewer re-ran this on a fresh Codespace, would they get the same reject count? Why?

In [9]:
logs.decision(table="bookings_legacy",
              found="21 repeated header lines inside appended exports",
              decision="drop",
              reason="They are repeated column headings, not booking records.",
               effect_on_headline="No booking records removed; no direct effect on the headline.",
              reversible="yes",
              rows_affected="21 header lines",
              owner="chaitanya", n=1)

logs.decision(table="bookings_all",
              found="3,107 booking records failed model validation; 564,879 passed",
              decision="split",
              reason="Keep failures in the rejects file with their validation reasons; only accepted bookings enter the validated booking table.",
               effect_on_headline="The final effect on the Brief A metric must be measured after the cleaning and reconciliation decisions.",
              reversible="yes",
              rows_affected="3,107 rejected",
              owner="chaitanya", n=2)


In [12]:
logs.decision(
    table="bookings_all",
    found="1,500 accepted Nova records have rescheduled_from populated",
    decision="keep and flag",
    reason=(
        "Preserve the reschedule information for the reschedule decision "
        "in notebook 03."
    ),
    rows_affected="1,500 flagged",
    effect_on_headline=(
        "Not decided here; notebook 03 will determine how reschedules affect the count."
    ),
    reversible="yes",
    owner="chaitanya",
    n=3,
)
logs.decision(
    table="bookings_all",
    found="The city label Bangalore was mapped to Bengaluru",
    decision="map",
    reason="Use one consistent city label for grouping and reporting.",
    rows_affected="Bangalore entries; count not recorded",
    effect_on_headline=(
        "No change to total bookings; city-level grouping may change."
    ),
    reversible="yes",
    owner="chaitanya",
    n=4,
)

In [13]:
logs.approach(
    part="How to read the booking files",
    considered="Read files sequentially; read files with a process pool",
    chosen="Use a process pool for the full load",
    why=(
        "The pack has many compressed CSV and JSON files that can be processed "
        "independently."
    ),
    would_change=(
        "A full sequential run that was faster or used less memory would favour "
        "sequential loading."
    ),
    happened=(
        "For 12 legacy files, sequential took 0.8 seconds and the process pool "
        "took 1.0 second. The full pool load took 2.3 seconds for legacy and "
        "0.7 seconds for Nova. A full sequential comparison was not run."
    ),
    owner="chaitanya",
    n=2,
)

logs.approach(
    part="How to handle bookings that fail validation",
    considered=(
        "Accept all loaded records; validate each booking and retain failures "
        "with reasons"
    ),
    chosen="Validate each booking and save rejected records with their reasons",
    why=(
        "This defines what enters the accepted booking table while keeping "
        "failures available for review."
    ),
    would_change=(
        "Evidence that valid real visits were being rejected would require "
        "revisiting the validation rules."
    ),
    happened=(
        "567,986 distinct input bookings reconciled to 564,879 accepted and "
        "3,107 rejected. The reconciliation check passed."
    ),
    owner="chaitanya",
    n=3,
)